# Dar_032 — Native terrain/illumination support checks

Port of original notebooks 11/12/25, using the existing local preparation, original maps and documented unit policy. Questions: are native crops decoded/aligned correctly; do terrain exclusions exactly follow the recorded support policy; do missing terrain and permanent shadow coincide? These are static spatial controls—not dated mission illumination/contact histories.


In [1]:
from pathlib import Path
import sys,json
sys.dont_write_bytecode=True
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'outline.md').is_file() and (p/'notebooks').is_dir())
sys.path[:0]=[str(ROOT / "env_algo_task/EDA/shared"), str(ROOT / "env_algo_task/EDA/Moon/Stage3_Requirements"),str(ROOT)]
import pandas as pd
import rasterio
from IPython.display import display
from environment_workflow import LocalRun
import terrain_followup_eda as methods
REGION_LIMIT=None


In [2]:
stage='analysis/stage3_native_support' if REGION_LIMIT is None else 'verification/stage3_native_support_sample'
with rasterio.Env(GDAL_CACHEMAX=32*1024**2):
    with LocalRun(ROOT,stage,{'source':'root notebooks 11/12/25','region_limit':REGION_LIMIT,'source_run':'20261106T180253772848Z_40d3f3','history_type':'static maps; no dated terrain-aware history'},'Moon/Stage3_Requirements/Dar_032_native_terrain_illumination_support.ipynb') as run:
        findings=methods.native_environment_port(run,ROOT,REGION_LIMIT)
        run.path('findings.json').write_text(json.dumps(findings,indent=2)+'\n',encoding='utf-8')
OUT=run.directory
display(findings)


Native/static support checks passed: A3CLR22_10_Leibnitz_Beta_Plateau
Native/static support checks passed: A3CLR22_11_Malapert_Massif
Native/static support checks passed: A3CLR22_12_deGerlache_Kocher_Massif
Native/static support checks passed: A3CLR22_2_Haworth
Native/static support checks passed: A3CLR22_3_Faustini_Rim_A
Native/static support checks passed: A3CLR22_4_deGerlache_Rim_2
Native/static support checks passed: A3CLR22_5_Connecting_Ridge_Extension


{'regions': 7,
 'source_crop_checks': 28,
 'native_coordinate_samples': 1792,
 'terrain_policy_mismatches': 0,
 'dated_terrain_aware_history_found': False,
 'limits': ['Static mean maps and PSR are terrain-aware spatial context, not dated mission histories',
  '201608 is a product identifier/date, not a per-interval timestamp',
  'Image-support exclusion is not equivalent to permanent shadow',
  'No full raw/orthomosaic gallery or original CNN model replay claimed']}

In [3]:
for name in ['terrain_policy_consistency.csv','native_source_crop_checks.csv','terrain_environment_support.csv']:
    print(name)
    display(pd.read_csv(OUT/name))
print('Outputs:',OUT)
assert not (ROOT/'data').exists()


terrain_policy_consistency.csv


,region,analysis_cells,elevation_policy_mismatches,slope_deg_neighbourhood_mismatches,laplacian_per_m_neighbourhood_mismatches,local_elevation_std_m_neighbourhood_mismatches
0,A3CLR22_10_Leibnitz_Beta_Plateau,284089,0,0,0,0
1,A3CLR22_11_Malapert_Massif,490000,0,0,0,0
2,A3CLR22_12_deGerlache_Kocher_Massif,443556,0,0,0,0
3,A3CLR22_2_Haworth,986049,0,0,0,0
4,A3CLR22_3_Faustini_Rim_A,444222,0,0,0,0
5,A3CLR22_4_deGerlache_Rim_2,444889,0,0,0,0
6,A3CLR22_5_Connecting_Ridge_Extension,444889,0,0,0,0


native_source_crop_checks.csv


,region,variable,unit,sample_positions,finite_pairs,maximum_absolute_difference,native_resolution_m
0,A3CLR22_10_Leibnitz_Beta_Plateau,solar_visibility,fraction,64,64,4.192352e-08,60.0
1,A3CLR22_10_Leibnitz_Beta_Plateau,earth_visibility,fraction,64,64,4.968643e-08,60.0
2,A3CLR22_10_Leibnitz_Beta_Plateau,permanent_shadow,binary,64,64,0.000000e+00,60.0
3,A3CLR22_10_Leibnitz_Beta_Plateau,sky_view_sr,sr,64,64,4.547119e-07,240.0
4,A3CLR22_11_Malapert_Massif,solar_visibility,fraction,64,64,3.082275e-08,60.0
5,A3CLR22_11_Malapert_Massif,earth_visibility,fraction,64,64,4.053116e-08,60.0
6,A3CLR22_11_Malapert_Massif,permanent_shadow,binary,64,64,0.000000e+00,60.0
7,A3CLR22_11_Malapert_Massif,sky_view_sr,sr,64,64,4.455566e-07,240.0
8,A3CLR22_12_deGerlache_Kocher_Massif,solar_visibility,fraction,64,64,2.883911e-08,60.0
9,A3CLR22_12_deGerlache_Kocher_Massif,earth_visibility,fraction,64,64,4.407883e-08,60.0


terrain_environment_support.csv


,region,cohort,analysis_grid_cells,known_psr_cells,psr_cells,scope,solar_visibility_valid_cells,solar_visibility_median,earth_visibility_valid_cells,earth_visibility_median,sky_view_sr_valid_cells,sky_view_sr_median
0,A3CLR22_10_Leibnitz_Beta_Plateau,all_roi,284089,284089,8628,"30 m analysis grid; 60/240 m source maps, not ...",284089,0.25996,284089,0.48712,284089,5.8402
1,A3CLR22_10_Leibnitz_Beta_Plateau,excluded_elevation,18420,18420,8440,"30 m analysis grid; 60/240 m source maps, not ...",18420,0.00248,18420,0.33264,18420,5.5414
2,A3CLR22_10_Leibnitz_Beta_Plateau,retained_elevation,265669,265669,188,"30 m analysis grid; 60/240 m source maps, not ...",265669,0.27332,265669,0.52144,265669,5.8530
3,A3CLR22_11_Malapert_Massif,all_roi,490000,490000,3700,"30 m analysis grid; 60/240 m source maps, not ...",490000,0.37900,490000,0.45580,490000,5.6824
4,A3CLR22_11_Malapert_Massif,excluded_elevation,12213,12213,3420,"30 m analysis grid; 60/240 m source maps, not ...",12213,0.03596,12213,0.00000,12213,5.7422
5,A3CLR22_11_Malapert_Massif,retained_elevation,477787,477787,280,"30 m analysis grid; 60/240 m source maps, not ...",477787,0.38212,477787,0.54216,477787,5.6818
6,A3CLR22_12_deGerlache_Kocher_Massif,all_roi,443556,443556,7306,"30 m analysis grid; 60/240 m source maps, not ...",443556,0.24172,443556,0.29800,443556,5.7130
7,A3CLR22_12_deGerlache_Kocher_Massif,excluded_elevation,16318,16318,6838,"30 m analysis grid; 60/240 m source maps, not ...",16318,0.00216,16318,0.25432,16318,5.2774
8,A3CLR22_12_deGerlache_Kocher_Massif,retained_elevation,427238,427238,468,"30 m analysis grid; 60/240 m source maps, not ...",427238,0.24556,427238,0.30256,427238,5.7164
9,A3CLR22_2_Haworth,all_roi,986049,986049,18184,"30 m analysis grid; 60/240 m source maps, not ...",986049,0.23692,986049,0.40716,986049,5.8206


Outputs: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\env_algo_task\data\processed\analysis\stage3_native_support\20261109T143112806464Z_f22d8c


## Interpretation

Reuses root scale/offset-once, native-coordinate and exact-neighbourhood checks. A 30 m comparison lattice does not create independent 30 m illumination measurements from 60/240 m sources. Missing image-supported terrain is not automatically permanent shadow. The 201608 filename identifies a product, not a sequence of contact/power states. No endurance or communication-continuity pass follows.
